In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def obter_diretorio_input() -> Path:
    if "INPUT_DIR" in os.environ and os.path.isdir(os.environ["INPUT_DIR"]):
        return Path(os.environ["INPUT_DIR"])
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / "workflows").is_dir():
            return p / "workflows"
        if (p / "hydranomic" / "workflows").is_dir():
            return p / "hydranomic" / "workflows"
    return Path.cwd()

def obter_diretorio_output() -> Path:
    if "OUTPUT_DIR" in os.environ:
        caminho = Path(os.environ["OUTPUT_DIR"])
        caminho.mkdir(parents=True, exist_ok=True)
        return caminho
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / "workflows").is_dir():
            return p / "workflows"
        if (p / "hydranomic" / "workflows").is_dir():
            return p / "hydranomic" / "workflows"
    caminho = Path.cwd() / "outputs"
    caminho.mkdir(parents=True, exist_ok=True)
    return caminho

def carregar_json(nome_arquivo: str, padrao: Any = None) -> Any:
    dir_in = obter_diretorio_input()
    candidatos = [
        dir_in / nome_arquivo,
        Path.cwd() / "workflows" / nome_arquivo,
        Path.cwd() / "hydranomic" / "workflows" / nome_arquivo,
        Path.cwd() / nome_arquivo
    ]
    for c in candidatos:
        if c.exists() and c.stat().st_size > 0:
            with open(c, "r", encoding="utf-8") as f:
                return json.load(f)
    if padrao is not None:
        return padrao
    raise FileNotFoundError(f"Arquivo {nome_arquivo} nao encontrado em {dir_in}")

def salvar_json(nome_arquivo: str, dados: Any, indent: int = 2) -> Path:
    dir_out = obter_diretorio_output()
    caminho = dir_out / nome_arquivo
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=indent)
    return caminho

def carregar_contexto() -> Dict[str, Any]:
    try:
        return carregar_json("context.json", padrao={})
    except Exception:
        return {}

# Aliases de compatibilidade
ler_json = carregar_json
gravar_json = salvar_json
ler_contexto_wegnology = carregar_contexto
get_input_dir = obter_diretorio_input
get_output_dir = obter_diretorio_output

print(f"✓ Ambiente pronto!")
print(f"  - INPUT_DIR:  {get_input_dir()}")
print(f"  - OUTPUT_DIR: {get_output_dir()}")


--- 
## Bloco 2: Parâmetros Injetáveis (Papermill / WEGnology Workflow)
*(Esta célula pode ser sobrescrita automaticamente durante a execução agendada no WEGnology)*

In [ ]:
ESTADO = "tudo"
TARIFA_AGUA = 59.6
ARQUIVO_HISTORICO = "conta.json"
N_ESTIMATORS = 250
ANO_GRAFICO = None
ESTADO_DESEJADO = ESTADO
PERSISTIR_ARQUIVOS = True
ANO_REFERENCIA_GRAFICO = ANO_GRAFICO


In [ ]:
def gerenciar_consumo_diario_e_contas(contexto: Optional[Dict[str, Any]] = None) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    if contexto is None:
        contexto = carregar_contexto()
    if "body" in contexto and isinstance(contexto["body"], dict):
        for k, v in contexto["body"].items():
            if k not in contexto:
                contexto[k] = v
    hoje = datetime.now()
    mes_atual_id = f"{hoje.year}-{hoje.month:02d}"
    data_hoje_str = hoje.strftime("%Y-%m-%d")
    consumo_mes_padrao = {
        "mes_referencia": mes_atual_id,
        "unidade_consumidora": 14079,
        "consumo_acumulado_mes_m3": 0.0,
        "dias": []
    }
    try:
        consumo_mes = carregar_json("consumo_mes.json", padrao=consumo_mes_padrao)
    except Exception:
        consumo_mes = consumo_mes_padrao
    try:
        dados_contas = carregar_json("conta.json")
    except Exception:
        try:
            dados_contas = carregar_json("conta_povoada.json")
        except Exception:
            dados_contas = []

    mes_salvo = str(consumo_mes.get("mes_referencia", mes_atual_id))
    if mes_salvo != mes_atual_id:
        acumulado_anterior = float(consumo_mes.get("consumo_acumulado_mes_m3", 0.0))
        if acumulado_anterior == 0.0 and consumo_mes.get("dias"):
            acumulado_anterior = sum(float(d.get("consumo_m3", 0.0)) for d in consumo_mes.get("dias", []))
        if acumulado_anterior > 0:
            encontrado_ant = False
            for idx, r in enumerate(dados_contas):
                if str(r.get("id")) == mes_salvo:
                    dados_contas[idx]["consumo_m3"] = round(acumulado_anterior, 2)
                    if "valor_total" not in dados_contas[idx]:
                        dados_contas[idx]["valor_total"] = round(acumulado_anterior * TARIFA_AGUA, 2)
                    encontrado_ant = True
                    break
            if not encontrado_ant:
                dados_contas.append({
                    "id": mes_salvo,
                    "unidade_consumidora": consumo_mes.get("unidade_consumidora", 14079),
                    "categoria": "COMERCIAL",
                    "consumo_m3": round(acumulado_anterior, 2),
                    "valor_total": round(acumulado_anterior * TARIFA_AGUA, 2)
                })
        consumo_mes = {
            "mes_referencia": mes_atual_id,
            "unidade_consumidora": consumo_mes.get("unidade_consumidora", 14079),
            "consumo_acumulado_mes_m3": 0.0,
            "dias": []
        }

    dias = consumo_mes.get("dias", [])
    consumo_dia = contexto.get("consumo_dia") or contexto.get("novo_consumo_dia") or contexto.get("consumo_hoje")
    if consumo_dia is None:
        pulsos_raw = None
        if "data" in contexto and isinstance(contexto["data"], dict):
            pulsos_raw = contexto["data"].get("Pulsos") or contexto["data"].get("pulsos")
        if pulsos_raw is None:
            pulsos_raw = contexto.get("Pulsos") or contexto.get("pulsos")
        if isinstance(pulsos_raw, dict):
            pulsos_raw = pulsos_raw.get("value")
        if pulsos_raw is not None:
            try:
                consumo_dia = round(float(pulsos_raw) * 0.001, 3)
            except (ValueError, TypeError):
                consumo_dia = None

    data_registro = contexto.get("data")
    if not data_registro and "time" in contexto and isinstance(contexto["time"], str):
        data_registro = contexto["time"].split("T")[0]
    if not data_registro or not isinstance(data_registro, str):
        data_registro = data_hoje_str

    try:
        dia_num = int(contexto.get("dia", data_registro.split("-")[-1]))
    except Exception:
        dia_num = hoje.day

    if consumo_dia is not None:
        try:
            consumo_float = float(consumo_dia)
            encontrado_dia = False
            for idx, d in enumerate(dias):
                if d.get("data") == data_registro or d.get("dia") == dia_num:
                    consumo_anterior = float(dias[idx].get("consumo_m3", 0.0) or 0.0)
                    dias[idx]["consumo_m3"] = round(consumo_anterior + consumo_float, 3)
                    dias[idx]["data"] = data_registro
                    dias[idx]["dia"] = dia_num
                    encontrado_dia = True
                    break
            if not encontrado_dia:
                dias.append({"dia": dia_num, "data": data_registro, "consumo_m3": consumo_float})
        except Exception:
            pass

    dias = sorted(dias, key=lambda x: int(x.get("dia", 0)))
    consumo_mes["dias"] = dias
    total_acumulado = round(sum(float(d.get("consumo_m3", 0.0)) for d in dias), 2)
    consumo_mes["consumo_acumulado_mes_m3"] = total_acumulado

    # Cálculo da vazão instantânea em m³/h
    ultima_leitura = consumo_mes.get("ultima_leitura", {}) if isinstance(consumo_mes, dict) else {}
    pulsos_anteriores = ultima_leitura.get("pulsos")
    time_anterior_str = ultima_leitura.get("timestamp")
    timestamp_anterior = None
    if time_anterior_str and isinstance(time_anterior_str, str):
        try:
            timestamp_anterior = datetime.fromisoformat(time_anterior_str.replace("Z", "+00:00")).replace(tzinfo=None)
        except Exception:
            timestamp_anterior = None

    vazao_m3h = 0.0
    timestamp_atual = hoje
    if "time" in contexto and isinstance(contexto["time"], str):
        try:
            timestamp_atual = datetime.fromisoformat(contexto["time"].replace("Z", "+00:00")).replace(tzinfo=None)
        except Exception:
            timestamp_atual = hoje

    pulsos_atuais = None
    if pulsos_raw is not None:
        try:
            pulsos_atuais = float(pulsos_raw)
        except Exception:
            pulsos_atuais = None

    if timestamp_anterior is not None:
        delta_seg = (timestamp_atual - timestamp_anterior).total_seconds()
        if pulsos_atuais is not None and pulsos_anteriores is not None and pulsos_atuais > pulsos_anteriores:
            delta_vol = (pulsos_atuais - pulsos_anteriores) * 0.001
        else:
            delta_vol = 0.1

        if 3 <= delta_seg <= 7200:
            vazao_m3h = round((delta_vol * 3600.0) / delta_seg, 2)
        else:
            vazao_m3h = 0.0
    else:
        vazao_m3h = float(consumo_mes.get("vazao_atual_m3h", 0.0) or 0.0)

    consumo_mes["vazao_atual_m3h"] = vazao_m3h
    consumo_mes["ultima_leitura"] = {
        "pulsos": pulsos_atuais if pulsos_atuais is not None else pulsos_anteriores,
        "timestamp": timestamp_atual.isoformat(),
        "vazao_calculada_m3h": vazao_m3h
    }

    salvar_json("consumo_mes.json", consumo_mes)

    nova_conta = contexto.get("nova_conta") or contexto.get("novo_registro")
    if nova_conta and isinstance(nova_conta, dict):
        id_novo = str(nova_conta.get("id", ""))
        encontrado = False
        for idx, r in enumerate(dados_contas):
            if str(r.get("id")) == id_novo:
                dados_contas[idx].update(nova_conta)
                encontrado = True
                break
        if not encontrado:
            dados_contas.append(nova_conta)
    elif total_acumulado > 0:
        encontrado_mes = False
        for idx, r in enumerate(dados_contas):
            if str(r.get("id")) == mes_atual_id:
                dados_contas[idx]["consumo_m3"] = total_acumulado
                if "valor_total" not in dados_contas[idx] or dados_contas[idx].get("categoria") == "COMERCIAL":
                    dados_contas[idx]["valor_total"] = round(total_acumulado * TARIFA_AGUA, 2)
                encontrado_mes = True
                break
        if not encontrado_mes:
            dados_contas.append({
                "id": mes_atual_id,
                "unidade_consumidora": consumo_mes.get("unidade_consumidora", 14079),
                "categoria": "COMERCIAL",
                "consumo_m3": total_acumulado,
                "valor_total": round(total_acumulado * TARIFA_AGUA, 2)
            })

    dados_contas = sorted(dados_contas, key=lambda x: str(x.get("id", "")))
    salvar_json("conta.json", dados_contas)
    return consumo_mes, dados_contas


In [ ]:
from datetime import datetime
from typing import Any, Dict, List, Optional
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def modulo_grafico_anual(
    dados_consumo: Optional[List[Dict[str, Any]]] = None,
    ano_referencia: Optional[int] = None,
    salvar: bool = True,
    exibir_grafico: bool = False
) -> Dict[str, Any]:
    """
    Processa os registros mensais e formata a série de 12 meses para o dashboard anual.
    Captura telemetria de pulsos em tempo de execução para atualizar o mês vigente.
    """
    # 1. Carrega dados de histórico
    if dados_consumo is None:
        try:
            dados_consumo = ler_json(ARQUIVO_HISTORICO)
        except Exception:
            try:
                dados_consumo = ler_json("conta.json", padrao=[])
            except Exception:
                dados_consumo = []

    # 2. Leitura do contexto injetado pelo WEGnology
    contexto = {}
    if "carregar_contexto" in globals():
        contexto = carregar_contexto()
    elif "ler_json" in globals():
        try:
            contexto = ler_json("context.json", padrao={})
        except Exception:
            contexto = {}

    # 3. Extração segura dos pulsos sem erro de int.get()
    pulsos_brutos = None
    if isinstance(contexto, dict):
        if "Pulsos" in contexto:
            pulsos_brutos = contexto["Pulsos"]
        elif "pulsos" in contexto:
            pulsos_brutos = contexto["pulsos"]
        elif isinstance(contexto.get("data"), dict):
            p_data = contexto["data"].get("Pulsos") or contexto["data"].get("pulsos")
            if isinstance(p_data, dict):
                pulsos_brutos = p_data.get("value")
            else:
                pulsos_brutos = p_data

        if isinstance(pulsos_brutos, dict):
            pulsos_brutos = pulsos_brutos.get("value")

    consumo_telemetria = None
    if pulsos_brutos is not None:
        try:
            consumo_telemetria = round(float(pulsos_brutos) / 1000.0, 2)
        except (ValueError, TypeError):
            consumo_telemetria = None
    else:
        consumo_direto = contexto.get("consumo_dia") or contexto.get("novo_consumo_dia") or contexto.get("consumo_hoje")
        if consumo_direto is not None:
            try:
                consumo_telemetria = round(float(consumo_direto), 2)
            except (ValueError, TypeError):
                consumo_telemetria = None

    hoje = datetime.now()
    mes_atual_id = f"{hoje.year}-{hoje.month:02d}"
    dia_num = hoje.day
    data_registro = hoje.strftime("%Y-%m-%d")
    consumo_float = consumo_telemetria

    # 4a. Armazenamento diário acumulativo em consumo_mes.json
    dados_mes = ler_json("consumo_mes.json", padrao={"mes": mes_atual_id, "dias": []}) if "ler_json" in globals() else {"mes": mes_atual_id, "dias": []}
    if not isinstance(dados_mes, dict):
        dados_mes = {"mes": mes_atual_id, "dias": []}
    dias = dados_mes.setdefault("dias", [])

    if consumo_float is not None:
        encontrado_dia = False
        for idx, d in enumerate(dias):
            if d.get("data") == data_registro or d.get("dia") == dia_num:
                # Acumula somando ao total já existente no dia
                consumo_anterior = float(d.get("consumo_m3", 0.0) or 0.0)
                dias[idx]["consumo_m3"] = round(consumo_anterior + consumo_float, 2)
                dias[idx]["data"] = data_registro
                dias[idx]["dia"] = dia_num
                encontrado_dia = True
                break
        if not encontrado_dia:
            dias.append({"dia": dia_num, "data": data_registro, "consumo_m3": round(consumo_float, 2)})
        dados_mes["mes_referencia"] = mes_atual_id
        dados_mes["consumo_acumulado_mes_m3"] = round(sum(float(d.get("consumo_m3", 0.0) or 0.0) for d in dias), 2)
        if salvar:
            if "salvar_json" in globals():
                salvar_json("consumo_mes.json", dados_mes)
            elif "gravar_json" in globals():
                gravar_json("consumo_mes.json", dados_mes)

    # 4b. Atualiza ou insere o consumo do mês atual (acumulando ao valor existente)
    if consumo_telemetria is not None:
        encontrado = False
        for reg in dados_consumo:
            if str(reg.get("id")) == mes_atual_id:
                consumo_anterior = float(reg.get("consumo_m3", 0.0) or 0.0)
                reg["consumo_m3"] = round(consumo_anterior + consumo_telemetria, 2)
                encontrado = True
                break
        if not encontrado:
            dados_consumo.append({
                "id": mes_atual_id,
                "consumo_m3": round(consumo_telemetria, 2),
                "unidade_consumidora": 14079
            })

    # 5. Define ano de referência
    if ano_referencia is None:
        anos = []
        for r in dados_consumo:
            partes = str(r.get("id", "")).split("-")
            if len(partes) == 2 and partes[0].isdigit():
                anos.append(int(partes[0]))
        ano_referencia = max(anos) if anos else hoje.year

    consumo_mensal = {}
    mes_atual = None

    for reg in dados_consumo:
        partes = str(reg.get("id", "")).split("-")
        if len(partes) == 2:
            try:
                a = int(partes[0])
                m = int(partes[1])
                if a == ano_referencia and 1 <= m <= 12:
                    consumo_mensal[m] = reg.get("consumo_m3")
                    mes_atual = m
            except ValueError:
                continue

    tags_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
    resultado_grafico = {
        "ano_referencia": ano_referencia,
        "mes_atual": mes_atual,
        "unidade_medida": "m³",
        "meses": [
            {
                "mes": mes,
                "rotulo": tags_meses[mes - 1],
                "consumo_m3": consumo_mensal.get(mes)
            }
            for mes in range(1, 13)
        ]
    }
    if salvar:
        if "salvar_json" in globals():
            salvar_json("grafico_anual.json", resultado_grafico)
        elif "gravar_json" in globals():
            gravar_json("grafico_anual.json", resultado_grafico)

    if exibir_grafico:
        labels = [m["rotulo"] for m in resultado_grafico["meses"]]
        vals = [m["consumo_m3"] if m["consumo_m3"] is not None else 0 for m in resultado_grafico["meses"]]
        cores = ["#0284c7" if m["consumo_m3"] is not None else "#e2e8f0" for m in resultado_grafico["meses"]]

        plt.figure(figsize=(9, 3.5))
        plt.bar(labels, vals, color=cores)
        plt.title(f"Consumo Mensal de Água - Ano {ano_referencia} (m³)")
        plt.xlabel("Mês")
        plt.ylabel("Consumo (m³)")
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.close()

    return resultado_grafico

# Teste autônomo da célula
ano_graf = globals().get("ANO_REFERENCIA_GRAFICO", datetime.now().year)
persistir = globals().get("PERSISTIR_ARQUIVOS", True)

res_grafico = modulo_grafico_anual(ano_referencia=ano_graf, salvar=persistir)
print(f"✓ Gráfico processado com sucesso para o ano {res_grafico['ano_referencia']}!")


--- 
## Bloco 4: Módulo de Machine Learning (`machinelearning`)
*(RandomForestRegressor com engenharia de atributos temporais e defasagem)*
*(Pode ser executado a qualquer momento de forma independente)*

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def executar_pipeline_ml(dados: Optional[List[Dict[str, Any]]] = None, tarifa: Optional[float] = None, salvar: bool = True) -> Dict[str, Any]:
    contexto = carregar_contexto()
    if tarifa is None:
        tarifa = float(contexto.get("tarifa", TARIFA_AGUA))
    if dados is None:
        _, dados = gerenciar_consumo_diario_e_contas(contexto)
    if not dados:
        try:
            dados = carregar_json("conta.json", padrao=[])
        except Exception:
            dados = []

    hoje = datetime.now()
    data_treinamento = hoje.isoformat()
    data_formatada = hoje.strftime("%d/%m/%Y %H:%M:%S")

    df = pd.DataFrame(dados) if dados else pd.DataFrame()
    if not df.empty and "id" in df.columns and "consumo_m3" in df.columns:
        df = df.dropna(subset=["id", "consumo_m3"]).copy()
        df = df.sort_values(by=["id"]).reset_index(drop=True)
        partes = df["id"].astype(str).str.split("-")
        df["ano"] = partes.str[0].astype(int)
        df["mes"] = partes.str[1].astype(int)
        if "unidade_consumidora" not in df.columns:
            df["unidade_consumidora"] = 14079

    try:
        if df.empty or len(df) < 2:
            raise ValueError("Historico insuficiente para modelo preditivo.")

        mes_atual_id = f"{hoje.year}-{hoje.month:02d}"
        df_fechados = df[df["id"] < mes_atual_id].copy()
        if len(df_fechados) < 2:
            df_fechados = df.copy()

        ano_inicial = int(df_fechados["ano"].min())
        mes_inicial = int(df_fechados.loc[df_fechados["ano"] == ano_inicial, "mes"].min())
        df_fechados["tempo_iniciofinal"] = (df_fechados["ano"] - ano_inicial) * 12 + (df_fechados["mes"] - mes_inicial)
        df_fechados["memoria_consumo"] = df_fechados.groupby("unidade_consumidora")["consumo_m3"].shift(1)
        df_treino = df_fechados.dropna(subset=["memoria_consumo"]).copy()

        features_entrada = ["mes", "tempo_iniciofinal", "memoria_consumo"]
        variavel_saida = "consumo_m3"
        modelo = RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=45)
        modelo.fit(df_treino[features_entrada], df_treino[variavel_saida])

        ultima_linha = df.iloc[-1]
        ultimo_ano = int(ultima_linha["ano"])
        ultimo_mes = int(ultima_linha["mes"])
        ultimo_consumo = float(ultima_linha["consumo_m3"])
        ultimo_passo = int((ultimo_ano - ano_inicial) * 12 + (ultimo_mes - mes_inicial))

        if ultimo_mes == 12:
            proximo_mes = 1
            proximo_ano = ultimo_ano + 1
        else:
            proximo_mes = ultimo_mes + 1
            proximo_ano = ultimo_ano
        proximo_passo = ultimo_passo + 1

        dados_previsao = pd.DataFrame([[proximo_mes, proximo_passo, ultimo_consumo]], columns=features_entrada)
        previsao_m3 = float(modelo.predict(dados_previsao)[0])
        valor_conta = round(previsao_m3 * tarifa, 2)
        resultado = {
            "ano": proximo_ano,
            "mes": proximo_mes,
            "unidade_consumidora": int(ultima_linha.get("unidade_consumidora", 14079)),
            "ultimo_consumo_registrado_m3": ultimo_consumo,
            "previsao_m3": round(previsao_m3, 2),
            "tarifa_m3": tarifa,
            "valor_estimado_conta": valor_conta,
            "modelo": "RandomForestRegressor",
            "data_treinamento": data_treinamento,
            "data_treinamento_formatada": data_formatada
        }
    except Exception as e:
        media_consumo = float(df["consumo_m3"].mean()) if not df.empty else 300.0
        ultimo_val = float(df.iloc[-1]["consumo_m3"]) if not df.empty else media_consumo
        uc = int(df.iloc[-1]["unidade_consumidora"]) if not df.empty and "unidade_consumidora" in df.columns else 14079
        mes_prox = (hoje.month % 12) + 1
        ano_prox = hoje.year if hoje.month < 12 else hoje.year + 1
        previsao_fallback = round((media_consumo + ultimo_val) / 2.0, 2)
        resultado = {
            "ano": ano_prox,
            "mes": mes_prox,
            "unidade_consumidora": uc,
            "ultimo_consumo_registrado_m3": round(ultimo_val, 2),
            "previsao_m3": previsao_fallback,
            "tarifa_m3": tarifa,
            "valor_estimado_conta": round(previsao_fallback * tarifa, 2),
            "modelo": "Fallback_MediaMovel",
            "aviso": str(e),
            "data_treinamento": data_treinamento,
            "data_treinamento_formatada": data_formatada
        }

    if salvar:
        salvar_json("previsao_ml.json", resultado)
    return resultado

modulo_machine_learning = executar_pipeline_ml


In [ ]:
def modulo_metas(
    consumo_atual_m3: Optional[float] = None,
    consumo_medido_m3: Optional[float] = None,
    salvar: bool = True
) -> Dict[str, Any]:
    """
    Lê config.json e calcula o progresso da meta ativa.
    Se consumo_atual_m3 não for passado, obtém o último valor dos dados históricos.
    Avalia alertas ativos de status (80%, estourada) e prazos (<= 2 dias) respeitando preferências.
    """
    if consumo_medido_m3 is None and consumo_atual_m3 is not None:
        consumo_medido_m3 = consumo_atual_m3

    # Carrega a configuração de metas
    config = ler_json("config.json", padrao={"meta_ativa": None, "metas_por_mes": {}, "solicitacao_exclusao": None, "notificacoes": {"notificacao_prazos": True, "alerta_status": True}})
    if not isinstance(config, dict):
        config = {"meta_ativa": None, "metas_por_mes": {}, "solicitacao_exclusao": None, "notificacoes": {"notificacao_prazos": True, "alerta_status": True}}
    if "notificacoes" not in config or not isinstance(config["notificacoes"], dict):
        config["notificacoes"] = {"notificacao_prazos": True, "alerta_status": True}
    config["notificacoes"].setdefault("notificacao_prazos", True)
    config["notificacoes"].setdefault("alerta_status", True)

    # Processa definição de meta ou notificações recebidas via contexto do Workflow
    try:
        ctx = carregar_contexto() if "carregar_contexto" in globals() else {}
        if isinstance(ctx, dict):
            if "body" in ctx and isinstance(ctx["body"], dict):
                for k, v in ctx["body"].items():
                    if k not in ctx:
                        ctx[k] = v

            # Atualização de preferências de notificação
            if ctx.get("acao") == "salvar_notificacoes" or "notificacao_prazos" in ctx or "alerta_status" in ctx or "notificacoes" in ctx:
                notifs = ctx.get("notificacoes")
                if isinstance(notifs, dict):
                    if "notificacao_prazos" in notifs:
                        config["notificacoes"]["notificacao_prazos"] = bool(notifs["notificacao_prazos"])
                    if "alerta_status" in notifs:
                        config["notificacoes"]["alerta_status"] = bool(notifs["alerta_status"])
                if "notificacao_prazos" in ctx:
                    config["notificacoes"]["notificacao_prazos"] = bool(ctx["notificacao_prazos"])
                if "alerta_status" in ctx:
                    config["notificacoes"]["alerta_status"] = bool(ctx["alerta_status"])

            if (ctx.get("acao") == "definir_meta" or ctx.get("nome")) and ctx.get("valor") is not None:
                periodo_meta = str(ctx.get("periodo") or ctx.get("data") or datetime.now().strftime("%Y-%m")).strip()
                mes_ref = periodo_meta[:7] if len(periodo_meta) >= 7 else datetime.now().strftime("%Y-%m")
                tipo_meta = str(ctx.get("tipo") or "consumo-mensal").strip()
                valor_meta = float(ctx.get("valor"))
                valor_litros = float(ctx.get("valor_litros")) if ctx.get("valor_litros") is not None else (valor_meta * 1000.0 if tipo_meta in ("consumo-mensal", "reducao_mensal") else None)
                nova_meta_obj = {
                    "tipo": tipo_meta,
                    "unidade": ctx.get("unidade") or ("L" if tipo_meta in ("consumo-mensal", "reducao_mensal") else "R$"),
                    "nome": ctx.get("nome", "Meta de Consumo"),
                    "valor": valor_meta,
                    "valor_litros": valor_litros,
                    "periodo": periodo_meta,
                    "mes_referencia": mes_ref,
                    "data_criacao": datetime.now().strftime("%Y-%m-%d"),
                    "descricao": ctx.get("descricao", ""),
                    "status": "ativa"
                }
                if not isinstance(config.get("metas_ativas_por_tipo"), dict):
                    config["metas_ativas_por_tipo"] = {}
                config["metas_ativas_por_tipo"][tipo_meta] = nova_meta_obj
                if tipo_meta in ("consumo-mensal", "reducao_mensal") or not config.get("meta_ativa"):
                    config["meta_ativa"] = nova_meta_obj
                if not isinstance(config.get("metas_por_mes"), dict):
                    config["metas_por_mes"] = {}
                reg_mes = config["metas_por_mes"].get(mes_ref)
                if not isinstance(reg_mes, dict):
                    reg_mes = {}
                if not isinstance(reg_mes.get("por_tipo"), dict):
                    reg_mes["por_tipo"] = {}
                reg_mes["por_tipo"][tipo_meta] = nova_meta_obj
                if tipo_meta in ("consumo-mensal", "reducao_mensal") or "valor" not in reg_mes:
                    reg_mes["nome"] = nova_meta_obj["nome"]
                    reg_mes["valor"] = nova_meta_obj["valor"]
                    reg_mes["valor_litros"] = valor_litros
                    reg_mes["periodo"] = periodo_meta
                    reg_mes["tipo"] = tipo_meta
                config["metas_por_mes"][mes_ref] = reg_mes
                config["solicitacao_exclusao"] = None
    except Exception:
        pass
    gravar_json("config.json", config)

    # Se o consumo não for informado, busca o mais recente
    if consumo_medido_m3 is None:
        try:
            consumo_mes = carregar_json("consumo_mes.json")
            consumo_atual_m3 = float(consumo_mes.get("consumo_acumulado_mes_m3", 0.0))
        except Exception:
            try:
                dados = carregar_json("conta.json")
                consumo_atual_m3 = float(dados[-1].get("consumo_m3", 0.0))
            except Exception:
                consumo_atual_m3 = 0.0
    meta = config.get("meta_ativa") if isinstance(config, dict) else None
    if consumo_atual_m3 is None:
        consumo_atual_m3 = float(consumo_medido_m3 or 0.0)
    if consumo_medido_m3 is None:
        consumo_medido_m3 = float(consumo_atual_m3 or 0.0)

    import calendar
    hoje = datetime.now()
    ultimo_dia_mes = calendar.monthrange(hoje.year, hoje.month)[1]
    dias_restantes = max(0, ultimo_dia_mes - hoje.day)

    conf_notif = config.get("notificacoes", {"notificacao_prazos": True, "alerta_status": True})
    permite_status = conf_notif.get("alerta_status", True)
    permite_prazos = conf_notif.get("notificacao_prazos", True)

    alertas_ativos = []

    if not meta:
        situacao = "NORMAL"
        relatorio = {
            "meta_ativa": False,
            "mensagem": "Nenhuma meta de consumo ativa configurada no momento.",
            "consumo_atual_m3": consumo_medido_m3,
            "situacao": situacao
        }
    else:
        limite = float(meta.get("valor", 1.0))
        porcentagem = round((consumo_atual_m3 / limite) * 100, 2)
        saldo_restante = round(limite - consumo_atual_m3, 2)
        if porcentagem >= 100:
            situacao = "ESTOURADA"
        elif porcentagem >= 80:
            situacao = "ALERTA_80"
        else:
            situacao = "NORMAL"
        relatorio = {
            "meta_ativa": True,
            "meta_nome": meta.get("nome", "Meta de Consumo"),
            "limite_m3": limite,
            "consumo_atual_m3": consumo_atual_m3,
            "percentual_atingido": porcentagem,
            "saldo_restante_m3": saldo_restante,
            "situacao": situacao,
            "periodo": meta.get("periodo", ""),
            "mes_referencia": meta.get("mes_referencia", meta.get("periodo", "")[:7])
        }

        # 1. Alerta de Status da Meta (80% ou Ultrapassada)
        if permite_status:
            if situacao == "ESTOURADA":
                alertas_ativos.append({
                    "id": "meta_estourada",
                    "tipo": "limite_estourado",
                    "severidade": "perigo",
                    "titulo": "Meta Ultrapassada!",
                    "mensagem": f"O consumo de {consumo_atual_m3:.2f} m³ ultrapassou o limite estabelecido de {limite:.2f} m³ ({porcentagem:.1f}% atingido).",
                    "timestamp": hoje.isoformat()
                })
            elif situacao == "ALERTA_80":
                alertas_ativos.append({
                    "id": "meta_80",
                    "tipo": "alerta_80",
                    "severidade": "alerta",
                    "titulo": "Atenção: 80% da Meta Atingida",
                    "mensagem": f"O consumo atingiu {porcentagem:.1f}% da meta mensal ({consumo_atual_m3:.2f} m³ de {limite:.2f} m³).",
                    "timestamp": hoje.isoformat()
                })

    # 2. Alerta de Prazo Próximo (48h / <= 2 dias para o fechamento do mês)
    if permite_prazos and dias_restantes <= 2:
        msg_prazo = "Faltam menos de 24h para o encerramento do ciclo mensal da sua meta." if dias_restantes <= 1 else f"Faltam {dias_restantes} dias para o encerramento do ciclo mensal da sua meta."
        alertas_ativos.append({
            "id": "prazo_fim_mes",
            "tipo": "prazo_proximo",
            "severidade": "info",
            "titulo": "Prazo de Fechamento Próximo",
            "mensagem": msg_prazo,
            "timestamp": hoje.isoformat()
        })

    relatorio["alertas_ativos"] = alertas_ativos
    relatorio["notificacoes_config"] = conf_notif
    relatorio["dias_restantes_mes"] = dias_restantes

    if salvar:
        gravar_json("progresso_meta.json", relatorio)

    return relatorio

avaliar_progresso_meta = modulo_metas

# Teste autônomo da célula (funciona em qualquer ordem)
res_meta = modulo_metas(salvar=PERSISTIR_ARQUIVOS)
print("✓ Relatório de metas processado com sucesso!")
print(res_meta)


--- 
## Bloco 6: BLOCO CONTROLADOR PRINCIPAL (Orquestrador do Pipeline)
**Este é o bloco controlador central.** Ele pode:
- Ser acionado manualmente escolhendo o estado (`1`, `2`, `3` ou `'tudo'`).
- Ser acionado automaticamente por um **Workflow da WEGnology** que injeta `context.json` com `{"estado": 1}` ou `{"acao": "ml"}`.
- Executar cada módulo de forma coordenada e gravar o `sumario_execucao.json` no `OUTPUT_DIR` para consumo direto pelos dashboards.

In [ ]:
def executar_controlador(estado_solicitado: Any = None) -> Dict[str, Any]:
    """
    Controlador mestre do pipeline analítico do Hydranomic.
    
    Estados suportados:
      - 1 ou 'ml': Executa apenas a previsão de Machine Learning.
      - 2 ou 'grafico': Executa apenas o processamento do gráfico anual.
      - 3 ou 'metas': Executa apenas a avaliação de metas de consumo.
      - 'tudo' ou 0: Executa o pipeline completo (Gráfico + ML + Metas).
    """
    print("=" * 60)
    print("       INICIANDO BLOCO CONTROLADOR HYDRANOMIC WEGNOLOGY       ")
    print("=" * 60)

    # 1. Verifica se há contexto recebido da WEGnology
    contexto = carregar_contexto() if "carregar_contexto" in globals() else {}
    if contexto:
        print(f"[Contexto WEGnology Detectado]: {contexto}")

    # Captura telemetria dos pulsos sem erro de int.get()
    pulsos_brutos = None
    if isinstance(contexto, dict):
        if "Pulsos" in contexto:
            pulsos_brutos = contexto["Pulsos"]
        elif "pulsos" in contexto:
            pulsos_brutos = contexto["pulsos"]
        elif isinstance(contexto.get("data"), dict):
            p_data = contexto["data"].get("Pulsos") or contexto["data"].get("pulsos")
            if isinstance(p_data, dict):
                pulsos_brutos = p_data.get("value")
            else:
                pulsos_brutos = p_data

        if isinstance(pulsos_brutos, dict):
            pulsos_brutos = pulsos_brutos.get("value")

    consumo_telemetria = 0.0
    if pulsos_brutos is not None:
        try:
            consumo_telemetria = round(float(pulsos_brutos) / 1000.0, 2)
        except (ValueError, TypeError):
            consumo_telemetria = 0.0
    else:
        consumo_telemetria = float(contexto.get("consumo_dia") or contexto.get("consumo_mes") or 0.0)
        if consumo_telemetria == 0.0:
            try:
                cm = carregar_json("consumo_mes.json", padrao={})
                consumo_telemetria = float(cm.get("consumo_acumulado_mes_m3", 0.0) or 0.0)
            except Exception:
                consumo_telemetria = 0.0

    # Prioridade do estado: argumento da função > contexto WEGnology > variável global ESTADO_DESEJADO
    estado = contexto.get("estado") or contexto.get("acao") or estado_solicitado or ESTADO_DESEJADO

    resultado_execucao = {
        "estado_executado": estado,
        "timestamp": datetime.now().isoformat(),
        "outputs_gerados": [
            "conta.json",
            "consumo_mes.json",
            "grafico_anual.json",
            "previsao_ml.json",
            "progresso_meta.json",
            "config.json",
            "sumario_execucao.json"
        ],
    }

    # Despachador de Ações (Match/Case resiliente)
    match str(estado).lower().strip():
        case "1" | "ml":
            print("▶ Executando Estado 1: Predição de Consumo via Machine Learning...")
            resultado_ml = modulo_machine_learning(salvar=True)
            resultado_execucao["ml"] = resultado_ml
            resultado_execucao["outputs_gerados"].append("previsao_ml.json")
            print(f"  -> Previsão calculada: {resultado_ml['previsao_m3']} m³ | R$ {resultado_ml['valor_estimado_conta']:.2f}")

        case "2" | "grafico":
            print("▶ Executando Estado 2: Geração de Dados para Gráfico Anual...")
            ano_param = contexto.get("ano_referencia", ANO_REFERENCIA_GRAFICO)
            chart_res = modulo_grafico_anual(ano_referencia=ano_param, salvar=True, exibir_grafico=False)
            resultado_execucao["grafico"] = chart_res
            resultado_execucao["outputs_gerados"].append("grafico_anual.json")
            print(f"  -> Gráfico consolidado para o ano {chart_res['ano_referencia']}.")

        case "3" | "metas" | "definir_meta" | "salvar_notificacoes":
            print("▶ Executando Estado 3: Avaliação de Metas de Consumo...")
            resultado_meta = avaliar_progresso_meta(consumo_atual_m3=consumo_telemetria, salvar=True)
            resultado_execucao["metas"] = resultado_meta
            resultado_execucao["outputs_gerados"].append("progresso_meta.json")
            print(f"  -> Relatório de meta gerado: {resultado_meta}")

        case _:
            print("▶ Executando Pipeline Completo (Consolidação de Todos os Módulos)...")
            
            # 1. Gráfico Anual
            chart_res = modulo_grafico_anual(salvar=True, exibir_grafico=False)
            resultado_execucao["grafico"] = chart_res
            resultado_execucao["outputs_gerados"].append("grafico_anual.json")

            # 2. Machine Learning
            resultado_ml = modulo_machine_learning(salvar=True)
            resultado_execucao["ml"] = resultado_ml
            resultado_execucao["outputs_gerados"].append("previsao_ml.json")

            # 3. Avalia a meta com o valor real
            resultado_meta = avaliar_progresso_meta(consumo_atual_m3=consumo_telemetria, salvar=True)
            resultado_execucao["metas"] = resultado_meta
            resultado_execucao["outputs_gerados"].append("progresso_meta.json")

    # Gera o sumário com o valor real acumulado do mês
    try:
        cm_sumario = carregar_json("consumo_mes.json", padrao={})
        consumo_mes_acumulado = float(cm_sumario.get("consumo_acumulado_mes_m3", 0.0) or 0.0)
    except Exception:
        consumo_mes_acumulado = consumo_telemetria

    # Cálculo pontual de vazão instantânea (m³/h)
    vazao_atual_m3h = 0.0
    try:
        medidor_estado = carregar_json("estado_medidor.json", padrao={})
        agora_dt = datetime.now()
        ts_ant_str = medidor_estado.get("timestamp")
        pulsos_ant = float(medidor_estado.get("pulsos_acumulados", 0.0) or 0.0)
        pulsos_atuais_num = float(pulsos_brutos) if pulsos_brutos is not None else float(consumo_telemetria * 1000.0)

        if ts_ant_str and pulsos_atuais_num >= pulsos_ant:
            delta_s = max((agora_dt - datetime.fromisoformat(ts_ant_str)).total_seconds(), 1.0)
            delta_m3 = (pulsos_atuais_num - pulsos_ant) / 1000.0
            vazao_atual_m3h = round((delta_m3 / delta_s) * 3600.0, 2)
        elif pulsos_atuais_num > 0:
            vazao_atual_m3h = round((pulsos_atuais_num / 1000.0) * 60.0, 2)

        salvar_json("estado_medidor.json", {
            "timestamp": agora_dt.isoformat(),
            "pulsos_acumulados": pulsos_atuais_num,
            "vazao_m3h": vazao_atual_m3h
        })
    except Exception:
        vazao_atual_m3h = 0.0

    conf_atual = ler_json("config.json", padrao={}).get("notificacoes", {"notificacao_prazos": True, "alerta_status": True})

    sumario = {
        "status": "sucesso",
        "vazao_atual_m3h": vazao_m3h_sumario,
        "mes_atual": datetime.now().strftime("%Y-%m"),
        "consumo_mes_acumulado_m3": consumo_mes_acumulado,
        "previsao_proximo_mes_m3": resultado_ml.get("previsao_m3") if "resultado_ml" in locals() else 0.0,
        "fatura_estimada_reais": resultado_ml.get("valor_estimado_conta") if "resultado_ml" in locals() else 0.0,
        "data_treinamento_ml": datetime.now().isoformat(),
        "ano_grafico": datetime.now().year,
        "meta_status": resultado_meta.get("situacao") if ("resultado_meta" in locals() and resultado_meta) else "NORMAL",
        "vazao_atual_m3h": vazao_atual_m3h,
        "alertas_ativos": resultado_meta.get("alertas_ativos", []) if ("resultado_meta" in locals() and resultado_meta) else [],
        "notificacoes": (resultado_meta.get("notificacoes_config") if ("resultado_meta" in locals() and resultado_meta and "notificacoes_config" in resultado_meta) else conf_atual)
    }
    salvar_json("sumario_execucao.json", sumario)
    resultado_execucao["sumario"] = sumario
    resultado_execucao["outputs_gerados"].append("sumario_execucao.json")

    print("=" * 60)
    print("       EXECUÇÃO DO CONTROLADOR CONCLUÍDA COM SUCESSO!       ")
    print(f"       Arquivos gerados no OUTPUT_DIR: {len(resultado_execucao['outputs_gerados'])}")
    print("=" * 60)
    return resultado_execucao

bloco_controlador = executar_controlador

# Executa o controlador (Você pode passar 1, 2, 3 ou 'tudo')
res_controlador = executar_controlador(ESTADO_DESEJADO)


--- 
## Bloco 7: Verificação e Auditoria de Arquivos no `OUTPUT_DIR`
*(Lista todos os arquivos persistidos que estão prontos para sincronização com os **Application Files** da WEGnology)*

In [ ]:
resultado = globals().get("res_controlador") or executar_controlador()
dir_saida = obter_diretorio_output()
arquivos = list(dir_saida.glob("*.json"))
print(f"Execucao concluida. Arquivos no OUTPUT_DIR ({dir_saida}):")
for a in arquivos:
    print(f" - {a.name} ({a.stat().st_size} bytes)")
